# Vinlab

DAT158-prosjekt. Denne notebooken viser data, trening og feilanalyse.


In [ ]:
from pathlib import Path
import sys, csv, json
import numpy as np
ROOT = Path.cwd()
if not (ROOT / 'data/winequality-red.csv').exists():
    raise FileNotFoundError('Start notebooken frå vinlab-mappa.')
sys.path.insert(0, str(ROOT / 'src'))
from models import *
print('NumPy:', np.__version__)


NumPy: 2.3.5


## 1 Problem og metrikkar

Målet er å estimere smakspoeng frå elleve kjemiske målingar. Dette er regresjon. MAE er hovudmetrikken; RMSE, R² og delen innanfor eitt poeng gir utfyllande informasjon. Låg MAE er bra. Ein høg del innanfor eitt poeng kan også kome frå å alltid gjette ei vanleg vurdering.

In [ ]:
with (ROOT / 'data/winequality-red.csv').open() as f:
    reader = csv.reader(f, delimiter=';')
    columns = next(reader)
    raw = np.array(list(reader), dtype=float)
_, kept = np.unique(raw[:, :-1], axis=0, return_index=True)
data = raw[np.sort(kept)]
print('Originale rader:', len(raw), 'Unike målingar:', len(data))
print('Manglande/ikkje-endelege verdiar:', np.sum(~np.isfinite(raw)))
print('Kvalitet  Tal')
for value, n in zip(*np.unique(data[:, -1], return_counts=True)):
    print(f'{int(value):8} {n:4}')
print('Felt                         Min      Median       Maks')
for i, field in enumerate(columns[:-1]):
    print(f'{field:25} {data[:,i].min():8.4f} {np.median(data[:,i]):10.4f} {data[:,i].max():10.4f}')


Originale rader: 1599 Unike målingar: 1359
Manglande/ikkje-endelege verdiar: 0
Kvalitet  Tal
       3   10
       4   53
       5  577
       6  535
       7  167
       8   17
Felt                         Min      Median       Maks
fixed acidity               4.6000     7.9000    15.9000
volatile acidity            0.1200     0.5200     1.5800
citric acid                 0.0000     0.2600     1.0000
residual sugar              0.9000     2.2000    15.5000
chlorides                   0.0120     0.0790     0.6110
free sulfur dioxide         1.0000    14.0000    72.0000
total sulfur dioxide        6.0000    38.0000   289.0000
density                     0.9901     0.9967     1.0037
pH                          2.7400     3.3100     4.0100
sulphates                   0.3300     0.6200     2.0000
alcohol                     8.4000    10.2000    14.9000


## 2 Trening og validering

Identiske målingar blir fjerna før oppdeling. Seed 158 gir faste 60/20/20-delar. Skalering for ridge blir tilpassa berre på treningsdelen.

In [ ]:
from train import main
main()


Median validation MAE 0.6765
Ridge alpha=1 validation MAE 0.5223
Ridge alpha=10 validation MAE 0.5216
Ridge alpha=100 validation MAE 0.5208
Tree depth=3 validation MAE 0.5656
Tree depth=6 validation MAE 0.5525
Tree depth=10 validation MAE 0.573
Forest depth=5 validation MAE 0.5322
Forest depth=8 validation MAE 0.52
Forest depth=12 validation MAE 0.51
Selected: Forest depth=12 
Test: {'mae': 0.4737063232474455, 'rmse': 0.6227769805809023, 'r2': 0.4331783189912338, 'within_one': 0.9007352941176471}


In [4]:
report = json.loads((ROOT / 'results/metrics.json').read_text(encoding='utf-8'))
print('Modell                 Trening MAE  Val MAE  Val RMSE  Val R2  Val innanfor 1')
for run in report['candidates']:
    v = run['validation']
    print(f"{run['name']:22} {run['train']['mae']:10.3f} {v['mae']:8.3f} {v['rmse']:9.3f} {v['r2']:7.3f} {100*v['within_one']:12.1f}%")
print('Vald modell:', report['selected'])
print('Test:', json.dumps(report['test'], indent=2))
print('Baseline test:', json.dumps(report['baseline_test'], indent=2))


Modell                 Trening MAE  Val MAE  Val RMSE  Val R2  Val innanfor 1
Median                      0.677    0.676     0.895  -0.245         94.5%
Ridge alpha=1               0.507    0.522     0.675   0.293         87.5%
Ridge alpha=10              0.507    0.522     0.674   0.295         87.5%
Ridge alpha=100             0.511    0.521     0.672   0.298         88.2%
Tree depth=3                0.519    0.566     0.719   0.198         85.7%
Tree depth=6                0.439    0.552     0.714   0.208         85.3%
Tree depth=10               0.370    0.573     0.755   0.114         86.0%
Forest depth=5              0.449    0.532     0.672   0.298         88.2%
Forest depth=8              0.360    0.520     0.667   0.310         88.6%
Forest depth=12             0.300    0.510     0.663   0.318         88.6%
Vald modell: Forest depth=12
Test: {
  "mae": 0.4737063232474455,
  "rmse": 0.6227769805809023,
  "r2": 0.4331783189912338,
  "within_one": 0.9007352941176471
}
Baseline te

## 3 Feilanalyse

Sjå spesielt på dei sjeldne kvalitetane. Små utval gir usikre mål. Permutasjonsanalysen under bruker berre valideringsdata. Han viser prediktiv nytte, ikkje årsakssamanhengar.

In [5]:
print('Testfeil fordelt på faktisk kvalitet:')
for row in report['quality_slices']:
    print(row)
print('Permutasjonsviktigheit: auke i MAE på validering, fem repetisjonar')
for row in report['permutation_importance_validation']:
    print(row)
print('Ti største absolutte testfeil:')
for row in report['worst_errors']:
    print(row)


Testfeil fordelt på faktisk kvalitet:
{'quality': 3, 'n': 3, 'mae': 1.9795515234712473}
{'quality': 4, 'n': 9, 'mae': 1.2271799199666762}
{'quality': 5, 'n': 110, 'mae': 0.33515310995203457}
{'quality': 6, 'n': 112, 'mae': 0.39782532138757937}
{'quality': 7, 'n': 35, 'mae': 0.7463338679211013}
{'quality': 8, 'n': 3, 'mae': 1.439960868606702}
Permutasjonsviktigheit: auke i MAE på validering, fem repetisjonar
{'feature': 'alcohol', 'mae_increase': 0.09576125387146484}
{'feature': 'sulphates', 'mae_increase': 0.03949236727320882}
{'feature': 'volatile acidity', 'mae_increase': 0.026417505646919026}
{'feature': 'total sulfur dioxide', 'mae_increase': 0.01206643385942352}
{'feature': 'density', 'mae_increase': 0.006976410992063253}
{'feature': 'chlorides', 'mae_increase': 0.0032686950891097633}
{'feature': 'citric acid', 'mae_increase': 0.0017141611023708814}
{'feature': 'fixed acidity', 'mae_increase': 0.0009941970460410898}
{'feature': 'free sulfur dioxide', 'mae_increase': 0.000921044453